## Practico Aprendizaje No Supervisado

### Importar librerias

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

sns.set_theme(style="whitegrid")

### Cargar datos

In [24]:
male_players = pd.read_csv("../Data/male_players.csv")

C:\Users\Gonza Marengo\AppData\Local\Temp\ipykernel_14368\4161528585.py:1: DtypeWarning: Columns (0: gk) have mixed types. Specify dtype option on import or set low_memory=False.
  male_players = pd.read_csv("../Data/male_players.csv")


### Estructura del dataset y tipos de variables

El dataset analizado contiene *'180.021 filas'* y *'109 columnas'*

In [28]:
print("Dimensiones del dataset: ", male_players.shape)
print(male_players.info())

Dimensiones del dataset:  (180021, 109)
<class 'pandas.DataFrame'>
RangeIndex: 180021 entries, 0 to 180020
Columns: 109 entries, player_id to gk
dtypes: float64(20), int64(43), object(1), str(45)
memory usage: 149.7+ MB
None


### Limpieza de datos

Hacemos una copia del dataset original para no modificar el dataset original.

Notamos que las columnas con atributos de jugadores como 'ls', 'st', 'rs', son variables de tipo string, cuando en realidad deberian ser numericas, hay valores que son por ejemplo **90+3**. Procedemos a limpiar estas 27 variables y asignarle su verdadero data type.

In [32]:
# crear la copia ANTES de tocar nada
male_players_limpio = male_players.copy()

# 1. Detectar columnas afectadas (sobre la copia)
afectadas = [c for c in male_players_limpio.columns
             if male_players_limpio[c].astype(str).str.replace(" ", "", regex=False)
                                                   .str.match(r'^\d+[+-]\d+$').any()]
print("Columnas a convertir:", afectadas)
print("Cantidad de columnas a convertir:", len(afectadas))

Columnas a convertir: ['ls', 'st', 'rs', 'lw', 'lf', 'cf', 'rf', 'rw', 'lam', 'cam', 'ram', 'lm', 'lcm', 'cm', 'rcm', 'rm', 'lwb', 'ldm', 'cdm', 'rdm', 'rwb', 'lb', 'lcb', 'cb', 'rcb', 'rb', 'gk']
Cantidad de columnas a convertir: 27


In [33]:
# 2. Chequeo de seguridad
for c in afectadas:
    no_nulos_antes = male_players_limpio[c].notna().sum()
    parseables     = male_players_limpio[c].map(parse_rating).notna().sum()
    perdidos = no_nulos_antes - parseables
    if perdidos > 0:
        print(f"  ⚠ {c}: {perdidos} valores no parseables (revisar antes de seguir)")
    else:
        print(f"  ✓ {c}: OK")

  ✓ ls: OK
  ✓ st: OK
  ✓ rs: OK
  ✓ lw: OK
  ✓ lf: OK
  ✓ cf: OK
  ✓ rf: OK
  ✓ rw: OK
  ✓ lam: OK
  ✓ cam: OK
  ✓ ram: OK
  ✓ lm: OK
  ✓ lcm: OK
  ✓ cm: OK
  ✓ rcm: OK
  ✓ rm: OK
  ✓ lwb: OK
  ✓ ldm: OK
  ✓ cdm: OK
  ✓ rdm: OK
  ✓ rwb: OK
  ✓ lb: OK
  ✓ lcb: OK
  ✓ cb: OK
  ✓ rcb: OK
  ✓ rb: OK
  ✓ gk: OK


In [34]:
# 3. Aplicar la conversión SOBRE LA COPIA
for c in afectadas:
    male_players_limpio[c] = pd.to_numeric(male_players_limpio[c].map(parse_rating)).astype("Int64")

# 4. Verificar
print(male_players_limpio[afectadas].dtypes)
male_players_limpio[afectadas].head()

ls     Int64
st     Int64
rs     Int64
lw     Int64
lf     Int64
cf     Int64
rf     Int64
rw     Int64
lam    Int64
cam    Int64
ram    Int64
lm     Int64
lcm    Int64
cm     Int64
rcm    Int64
rm     Int64
lwb    Int64
ldm    Int64
cdm    Int64
rdm    Int64
rwb    Int64
lb     Int64
lcb    Int64
cb     Int64
rcb    Int64
rb     Int64
gk     Int64
dtype: object


,ls,st,rs,lw,lf,cf,rf,rw,lam,cam,...,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk
0,93,93,93,91,91,91,91,91,92,92,...,66,66,66,71,66,57,57,57,66,21
1,93,93,93,82,86,86,86,82,85,85,...,66,66,66,65,63,65,65,65,63,22
2,86,86,86,87,88,88,88,87,91,91,...,83,83,83,82,78,73,73,73,78,24
3,88,88,88,90,89,89,89,90,90,90,...,66,66,66,67,62,52,52,52,62,22
4,90,90,90,86,89,89,89,86,90,90,...,67,67,67,67,63,58,58,58,63,21


In [35]:
male_players_limpio.head()

,player_id,player_url,fifa_version,fifa_update,update_as_of,short_name,long_name,player_positions,overall,potential,...,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk
0,231747,/player/231747/kylian-mbappe/240002,24.0,2.0,2023-09-22,K. Mbappé,Kylian Mbappé Lottin,"ST, LW",91,94,...,66,66,66,71,66,57,57,57,66,21
1,239085,/player/239085/erling-haaland/240002,24.0,2.0,2023-09-22,E. Haaland,Erling Braut Haaland,ST,91,94,...,66,66,66,65,63,65,65,65,63,22
2,192985,/player/192985/kevin-de-bruyne/240002,24.0,2.0,2023-09-22,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,...,83,83,83,82,78,73,73,73,78,24
3,158023,/player/158023/lionel-messi/240002,24.0,2.0,2023-09-22,L. Messi,Lionel Andrés Messi Cuccittini,"CF, CAM",90,90,...,66,66,66,67,62,52,52,52,62,22
4,165153,/player/165153/karim-benzema/240002,24.0,2.0,2023-09-22,K. Benzema,Karim Benzema,"CF, ST",90,90,...,67,67,67,67,63,58,58,58,63,21
